# 🎙️ Hima Hindi TTS (Quantized INT8)

Ultra-fast, lightweight Hindi Text-to-Speech using **Hima TTS Nano (62M) INT8 ONNX Quantization** by Prahalad Reddy.

- **Quantization:** INT8 (3.3x smaller, 15x real-time speed, high accuracy)
- **Built-in Voices:** Meera, Sneha, Nandini (Female) | Siddharth, Ishaan, Nikhil (Male) | Random
- **Text Support:** Devanagari Hindi + conversational English/Hinglish (`Weekend पे Goa का plan...`)

*(Runs instantly on any standard runtime - no GPU required!)*

## ⚙️ Step 1: Install & Load Quantized INT8 Model

In [ ]:
#@title 1. Setup & Load Model (Click Play ▶️)
!pip install -q onnxruntime gradio soundfile sentencepiece pyyaml safetensors huggingface_hub

from huggingface_hub import snapshot_download
import sys

print("📥 Downloading Hima Hindi TTS (INT8 Quantized)...")
snapshot_download(repo_id="Prahaladha/hima-tts-nano-hi-ONNX", local_dir="/content/hima-hi-onnx")

sys.path.insert(0, "/content/hima-hi-onnx")
from hima_onnx import OnnxTTS

print("⚡ Initializing INT8 quantized engine...")
tts = OnnxTTS("/content/hima-hi-onnx", precision="int8")
print("\n✅ Ready! Hindi TTS INT8 model is loaded and ready.")

## 🗣️ Step 2: Generate Hindi Speech

In [ ]:
#@title 2. Generate Audio (Click Play ▶️)
from IPython.display import Audio, display

hindi_text = "नमस्ते! आज का मौसम बहुत अच्छा है, आप कैसे हैं?" #@param {type:"string"}
voice = "Meera" #@param ["Meera", "Sneha", "Nandini", "Siddharth", "Ishaan", "Nikhil", "Random"]

selected_voice = voice if voice != "Random" else None
audio = tts.say(hindi_text, voice=selected_voice)
tts.save(audio, "/content/output.wav")

display(Audio("/content/output.wav", autoplay=True))

## 🌐 Step 3: Launch Interactive Web App (Gradio)

In [ ]:
#@title 3. Launch Web App (Click Play ▶️)
import gradio as gr

HINDI_VOICES = ["Meera", "Sneha", "Nandini", "Siddharth", "Ishaan", "Nikhil", "Random"]

def synthesize(text, voice_name):
    if not text or not text.strip():
        return None, "Please enter some Hindi text."
    v = voice_name if voice_name != "Random" else None
    audio = tts.say(text, voice=v)
    tts.save(audio, "/content/gradio_output.wav")
    return "/content/gradio_output.wav", f"Synthesized with voice: {voice_name}"

with gr.Blocks(title="Hima Hindi TTS Studio") as demo:
    gr.Markdown("# 🎙️ Hima Hindi Text-to-Speech (Quantized INT8)")
    gr.Markdown("Fast, natural Hindi speech generation with 6 built-in voice models.")
    
    with gr.Row():
        with gr.Column():
            txt = gr.Textbox(
                label="Hindi Text",
                value="यार, आज office में इतना काम था कि lunch करने का भी time नहीं मिला।",
                lines=4
            )
            v_choice = gr.Dropdown(HINDI_VOICES, value="Meera", label="Voice Model")
            btn = gr.Button("Generate Speech ▶️", variant="primary", size="lg")
        with gr.Column():
            aud = gr.Audio(label="Generated Audio", type="filepath")
            status = gr.Textbox(label="Status", lines=2)
            
    btn.click(synthesize, inputs=[txt, v_choice], outputs=[aud, status])

demo.launch(share=True)